In [ ]:
import itertools

import numpy as np
from scipy.io import loadmat

# File path, type pair, and selected measurement windows (stop is exclusive)
file_path = 'data_simulation/structure_function'
type_pair = ("A", "A")  # Two monomer names, including any underscores
iterations = range(100000, 500000, 100000)
file_names = [f"{file_path}_{step:06d}.mat" for step in iterations]
if not file_names:
    raise ValueError("Select at least one structure-function file.")

# Load the grid and cell from the first selected file: the lattice angles are saved with the
# structure functions, older files keep those of the input in initial_params
geometry = loadmat(file_names[0], squeeze_me=True, variable_names=['nx', 'lx', 'angles', 'initial_params'])
nx = np.atleast_1d(geometry['nx']).astype(int)
lx = np.atleast_1d(np.asarray(geometry['lx'], dtype=float))
dim = len(nx)
angles = np.array([90.0, 90.0, 90.0])
if 'angles' in geometry:
    angles = np.asarray(geometry['angles'], dtype=float)
elif 'initial_params' in geometry and 'angles' in (geometry['initial_params'].dtype.names or ()):
    input_angles = np.atleast_1d(np.asarray(geometry['initial_params']['angles'].item(), dtype=float))
    if input_angles.size == 3:
        angles = input_angles
    elif input_angles.size == 1:  # gamma of a 2D cell
        angles[2] = input_angles[0]
alpha, beta, gamma = np.deg2rad(angles)

# Lattice vectors as rows (a along x, b in the xy plane, as in ComputationBox), reduced to the
# dim dimensions of the grid, and the reciprocal metric G* = (L L^T)^-1:
# |k|^2 = (2 pi)^2 m^T G* m for the integer index m
cx = np.cos(beta)
cy = (np.cos(alpha) - np.cos(beta)*np.cos(gamma))/np.sin(gamma)
L = lx[:, None]*np.array([[1.0, 0.0, 0.0],
                          [np.cos(gamma), np.sin(gamma), 0.0],
                          [cx, cy, np.sqrt(1.0 - cx**2 - cy**2)]])[:dim, :dim]
G_star = np.linalg.inv(L @ L.T)

# Weight file means by the actual number of samples in each measurement window
sample_counts = []
for file_name in file_names:
    record = loadmat(file_name, variable_names=['sample_count'])
    if 'sample_count' not in record:
        sample_counts.append(None)
        continue
    count = np.asarray(record['sample_count'])
    if (count.size != 1 or count.dtype.kind not in 'iuf' or not np.isfinite(count).all()
            or count.item() < 0 or count.item() != np.floor(count.item())):
        raise ValueError(f"sample_count must be a finite nonnegative integer in {file_name}.")
    sample_counts.append(float(count.item()))
# Legacy files all receive equal weight; unknown counts cannot be mixed with known counts.
if all(count is None for count in sample_counts):
    sample_counts = [1.0] * len(file_names)
elif any(count is None for count in sample_counts):
    raise ValueError("Cannot mix structure-function files with and without sample_count.")
if not any(count > 0 for count in sample_counts):
    raise ValueError("The selected structure-function files contain no samples.")

# The files hold the real-FFT half spectrum along the last axis: multiplicity of each of its
# planes in the full Fourier spectrum
half = nx[-1]//2 + 1
mode_weights = np.full(half, 2.0)
mode_weights[0] = 1.0
if nx[-1] % 2 == 0:
    mode_weights[-1] = 1.0

# Calculate k²/(2π)² and group Fourier modes with equal magnitude. Each stored index stands for
# the alias m + a n with the smallest |k|, as the solvers choose it (docs/theory/NumericalMethods.md,
# Section 3.5). Only an axis with more than one grid point that the metric couples to another axis
# can have it beyond the centred index; |m_d + a_d n_d| <= sqrt(max f (L L^T)_dd) bounds the search
def metric_form(m):
    return sum(G_star[p, q]*m[p]*m[q] for p in range(dim) for q in range(dim))


m = np.meshgrid(*[np.arange(n) for n in nx[:-1]], np.arange(half), indexing='ij')
m = [np.where(m[d] <= nx[d]//2, m[d], m[d] - nx[d]) for d in range(dim - 1)] + [m[-1]]
k2 = metric_form(m)
coupled = [np.max(np.abs(np.delete(G_star[d], d)), initial=0.0) > 1e-12*np.max(np.abs(G_star))
           for d in range(dim)]
reach = [int(np.ceil(np.sqrt(k2.max()*(L @ L.T)[d, d])/nx[d])) + 1 if coupled[d] and nx[d] > 1 else 0
         for d in range(dim)]
for shift in itertools.product(*[range(-r, r + 1) for r in reach]):
    k2 = np.minimum(k2, metric_form([m[d] + shift[d]*nx[d] for d in range(dim)]))

# Shells: values equal to a relative 1e-9 (round-off of the metric), labelled by k²/(2π)²
# rounded to 7 decimals; shell[i] is the shell of the i-th stored mode
order = np.argsort(k2, axis=None)
k2_sorted = k2.ravel()[order]
first = np.concatenate(([True], np.diff(k2_sorted) > 1e-9*np.maximum(k2_sorted[1:], 1.0)))
k2_unique = np.round(k2_sorted[first], 7)
shell = np.empty(k2.size, dtype=int)
shell[order] = np.cumsum(first) - 1

sf_mag = np.zeros_like(k2_unique)
sf_count = np.zeros_like(k2_unique)
for file_name, sample_count in zip(file_names, sample_counts):
    if sample_count == 0:
        continue
    print(file_name)
    key = 'structure_function_' + '_'.join(sorted(type_pair))
    v = np.reshape(loadmat(file_name, variable_names=[key])[key], k2.shape)
    weight = np.broadcast_to(mode_weights*sample_count, k2.shape).ravel()
    sf_mag += np.bincount(shell, weights=v.real.ravel()*weight, minlength=k2_unique.size)
    sf_count += np.bincount(shell, weights=weight, minlength=k2_unique.size)

x = np.sqrt(k2_unique) * 2 * np.pi
y = sf_mag / sf_count
y[0] = 0.0  # Remove the k=0 mode


In [ ]:
import matplotlib.pyplot as plt

# Plot
plt.figure(dpi=100)
plt.style.use('classic')

plt.plot(x, y)
plt.title('Structure function')
plt.xlabel('$kR_0$')
plt.ylabel('$S(k)ρ_0N$')
plt.yscale('log')
plt.xlim([2, 14])
plt.ylim([0.01, 1000])
plt.grid(True)
plt.show()

In [ ]:
import pandas

# Write the curve as text (CSV)
df = pandas.DataFrame({'k': x, 's(k)': y})
df.to_csv("structure_function.csv", index=False)